# SURVEY DATA ANALYSIS

## 1. IMPORTS ALL NECESSARY LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import re
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

## 2. DATASET IMPORT AND CLEANING

In [ ]:
# imports dataset

path = "../data/Thesis.csv"
df_ = pd.read_csv(path)
df = df_.drop(index=[0,1]).reset_index(drop=True).copy()

# basic cleaning

for col in ["Finished", "Progress"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

if "Finished" in df.columns:
    df = df[df["Finished"].str.lower().isin(["true", "1", "yes"])]
if "Status" in df.columns:
    df = df[df["Status"].str.lower() != "survey preview"]

# renames Q*

rename_map = {}
for col in df.columns:
    if col.isdigit():
        rename_map[col] = f"Q{int(col)}"
    if "_TEXT" in col and col.split("_")[0].isdigit():
        q = col.split("_")[0]
        rename_map[col] = f"Q{q}_TEXT"
    if "_" in col and col.split("_")[0].isdigit() and not col.endswith("_TEXT"):
        q = col.split("_")[0]
        rest = "_".join(col.split("_")[1:])
        rename_map[col] = f"Q{q}_{rest}"

df = df.rename(columns=rename_map)
if "Q17" in df.columns: df = df.rename(columns={"Q17": "Q19_office"})
if "Q18" in df.columns: df = df.rename(columns={"Q18": "Q20_seniority"})

display("Data preview:", df.head(5))

## 2. DATA ANALYSIS
### 2.1 Q1 - Q2: Frequencies, crosstab, chi-squared

In [ ]:
output = "../outputs"
os.makedirs(output, exist_ok=True)

# definition of useful functions

# save function
def save_table(df, name):
    path = os.path.join(output, f"{name}.csv")
    df.to_csv(path, index=True)
    return path

# chi_square calculator
def chi_square_np(series, by):
    table = pd.crosstab(by, series, dropna=False)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return {"chi2": np.nan, "p_value": np.nan, "dof": np.nan}
    obs = table.values
    exp = (obs.sum(axis=1, keepdims=True) @ obs.sum(axis=0, keepdims=True)) / obs.sum()
    chi2 = np.nansum((obs-exp)**2/exp)
    dof = (obs.shape[0]-1)*(obs.shape[1]-1)
    try:
        from scipy.stats import chi2 as chi2_dist
        p = float(chi2_dist.sf(chi2, dof))
    except Exception:
        p = np.nan
    return {"chi2": float(chi2), "p_value": p, "dof": int(dof)}

# bar-plot builder
def bar_plot(series, title, fname):
    freq = series.value_counts(dropna=False).sort_index()
    plt.figure()
    freq.plot(kind="bar")
    plt.title(title); plt.ylabel("Count"); plt.xlabel("Response"); plt.tight_layout()
    plt.savefig(fname, dpi=150); plt.show()

# Q1 operations

if "Q1" in df.columns:
    q1 = df["Q1"]
    tab_q1 = q1.value_counts(dropna=False).rename_axis("Q1").to_frame("Count")
    tab_q1["Percent"] = (tab_q1["Count"]/len(q1)*100).round(2)
    save_table(tab_q1, "Q1_freq")
    bar_plot(q1, "Q1 – Projects with AI deliverables", os.path.join(output, "Q1_bar.png"))
    if "Q20_seniority" in df.columns:
        ct = pd.crosstab(df["Q20_seniority"], q1, dropna=False)
        save_table(ct, "Q1xQ20_counts")
        save_table(ct.div(ct.sum(axis=1), axis=0).mul(100).round(2), "Q1xQ20_rowpct")
        display("Q1 × Q20 (counts)", ct)
        display("Q1 × Q20 (row%)", ct.div(ct.sum(axis=1), axis=0).mul(100).round(2))

# Q2 operations 

if "Q2" in df.columns:
    q2 = df["Q2"]
    tab_q2 = q2.value_counts(dropna=False).rename_axis("Q2").to_frame("Count")
    tab_q2["Percent"] = (tab_q2["Count"]/len(q2)*100).round(2)
    save_table(tab_q2, "Q2_freq")
    bar_plot(q2, "Q2 – Client demand for AI (trend)", os.path.join(output, "Q2_bar.png"))
    if "Q19_office" in df.columns:
        ct2 = pd.crosstab(df["Q19_office"], q2, dropna=False)
        save_table(ct2, "Q2xQ19_counts")
        save_table(ct2.div(ct2.sum(axis=1), axis=0).mul(100).round(2), "Q2xQ19_rowpct")
        display("Q2 × Q19 (counts)", ct2)
        display("Q2 × Q19 (row%)", ct2.div(ct2.sum(axis=1), axis=0).mul(100).round(2))
        print("Chi-squared Q2×Q19:", chi_square_np(q2, df["Q19_office"]))

### 2.2 Multichoice (Q3, Q4, Q6, Q8, Q9, Q16)

In [ ]:
# definition of useful functions

# split function
def split_multiselect(s, sep=","):
    return s.dropna().astype(str).str.split(sep).explode().str.strip().pipe(lambda x: x[x!=""])

# counter
def multiselect_counts(s, title, basename):
    vals = split_multiselect(s)
    counts = vals.value_counts().to_frame("Count")
    counts["Percent of respondents"] = (counts["Count"]/s.notna().sum()*100).round(2)
    display(title+" – tabella", counts)
    plt.figure(figsize=(9,6))
    counts["Count"].iloc[::-1].plot(kind="barh")
    plt.title(title); plt.xlabel("Count"); plt.tight_layout()
    plt.savefig(os.path.join(output, f"{basename}_bar.png"), dpi=150); plt.show()
    counts.to_csv(os.path.join(output, f"{basename}_counts.csv"))

# Q3,Q4,Q6,Q8,Q9,Q16 operations
for q, name in [(3,"Q3 – Industries"), (4,"Q4 – Requested value"),
                (6,"Q6 – Most frequently used GPT tools"), (8,"Q8 – Barrier drivers"),
                (9,"Q9 – Perceived barriers"), (16,"Q16 – Desired GPT tools")]:
    col = f"Q{q}"
    if col in df.columns:
        multiselect_counts(df[col], name, f"Q{q}")

### 2.3 Q5 (ordinal) – Frequencies and seniority boxplot

In [ ]:
# calculates frequencies and plots results
if "Q5" in df.columns:
    order = ["Never","Rarely","A few times a week","Once a day","Several times a day"]
    mapping = {lab:i for i,lab in enumerate(order)}
    q5 = df["Q5"].dropna().astype(str)
    freq = q5.value_counts().reindex(order).fillna(0).astype(int).to_frame("Count")
    freq["Percent"] = (freq["Count"]/q5.shape[0]*100).round(2)
    display("Q5 – Usage frequency", freq)
    plt.figure(); freq["Count"].plot(kind="bar"); plt.title("Q5 – Usage frequency")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.savefig(os.path.join(output,"Q5_bar.png"), dpi=150); plt.show()
    
    # seniority box-plot
    if "Q20_seniority" in df.columns:
        tmp = pd.DataFrame({"score": df["Q5"].map(mapping), "seniority": df["Q20_seniority"]}).dropna()
        groups = [g["score"].values for _,g in tmp.groupby("seniority")]
        labels = [str(k) for k,_ in tmp.groupby("seniority")]
        plt.figure(figsize=(10,6)); plt.boxplot(groups, labels=labels, showmeans=True)
        plt.title("Q5 – Distribution by seniority (0=Never … 4=Several/day)")
        plt.xticks(rotation=45, ha="right"); plt.tight_layout()
        plt.savefig(os.path.join(output,"Q5_box_by_seniority.png"), dpi=150); plt.show()

### 2.4 Q7 – Heatmap: phase × seniority (mean 1–5)

In [ ]:
# reads question labels
raw_all = pd.read_csv(path)  
q7_cols = [c for c in df.columns if c.startswith("Q7_") and not c.endswith("_TEXT")]
# label mapping and mean calculation
if q7_cols:
    label_map = {}
    for qc in q7_cols:
        ok = qc.replace("Q","")
        label = str(raw_all.loc[0, ok]) if ok in raw_all.columns else qc
        label_map[qc] = label if label and label!="nan" else qc
    q7_df = df[q7_cols].apply(pd.to_numeric, errors="coerce").rename(columns=label_map)
    display("Q7 – Mean by phase", q7_df.mean().to_frame("Mean (1–5)").sort_values("Mean (1–5)"))

    # heatmap 
    if "Q20_seniority" in df.columns:
        mat = pd.concat([df["Q20_seniority"], q7_df], axis=1).groupby("Q20_seniority").mean()
        display("Q7 – Mean by phase × seniority", mat.round(2))
        fig, ax = plt.subplots(figsize=(10,6))
        im = ax.imshow(mat.values, aspect="auto")
        ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=45, ha="right")
        ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index)
        ax.set_title("Q7 – Heatmap: phase × GPT usage (mean)"); fig.colorbar(im, ax=ax, label="Media (1–5)")
        plt.tight_layout(); plt.savefig(os.path.join(output,"Q7_heatmap_phase_by_seniority.png"), dpi=150); plt.show()

### 2.5 New dataset for open answers

In [ ]:
text_cols = [c for c in df.columns if c.endswith("_TEXT") or c.lower().startswith("text")]
frames = []
for c in text_cols:
    s = df[c].dropna().astype(str)
    if not s.empty:
        tmp = pd.DataFrame({"Question": c, "Text": s})
        if "ResponseId" in df.columns:
            tmp["ResponseId"] = df.loc[s.index, "ResponseId"].values
        frames.append(tmp)
if frames:
    texts_df = pd.concat(frames, ignore_index=True)
    texts_df.to_csv(os.path.join(output, "free_text_responses.csv"), index=False)
    display("Open answers preview:", texts_df.head(50))

## 3. ADVANCED ANALYSIS

### 3.1 Useful function definition

In [ ]:
# labels cleaning
def sanitize_label(s):
    s = str(s)
    s = re.sub(r'\s*\(.*?\)\s*', '', s)  # removes parenthetical qualifiers such as "(specify)"
    s = re.sub(r'[^0-9A-Za-z]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    if s=='':
        s = 'empty'
    return s[:80]

# keep index while splitting
def split_multiselect_series(s, sep=','):
    s = s.fillna('').astype(str)
    out = {}
    for idx, val in s.items():
        if val.strip()=='':
            out[idx] = []
        else:
            items = [it.strip() for it in val.split(sep) if it.strip()!='']
            out[idx] = items
    return out

# series to df
def multiselect_dummies(s, prefix, sep=','):
    mapping = split_multiselect_series(s, sep=sep)
    choices = sorted({c for lst in mapping.values() for c in lst})
    cols = [f"{prefix}_{sanitize_label(c)}" for c in choices]
    df = pd.DataFrame(0, index=s.index, columns=cols)
    for idx, lst in mapping.items():
        for c in lst:
            col = f"{prefix}_{sanitize_label(c)}"
            if col in df.columns:
                df.at[idx, col] = 1
    return df

### 3.2 Feature engineering on Q5–Q13 for cluster and regression

In [ ]:
features = pd.DataFrame(index=df.index)

# orders the list if label are different
if "Q5" in df.columns:
    order_q5 = ["Never", "Rarely", "A few times a week", "Once a day", "Several times a day"]
    mapping_q5 = {lab:i for i,lab in enumerate(order_q5)}
    features['Q5_num'] = df['Q5'].map(mapping_q5)

# Q6, Q8, Q9, Q12: multichoice -> dummies
for q in ["Q6","Q8","Q9","Q12"]:
    if q in df.columns:
        dummies = multiselect_dummies(df[q], prefix=q)
        features = pd.concat([features, dummies], axis=1)

# add on Q7 sub-numbers (es. Q7_1, Q7_2, ...)
q7_cols = [c for c in df.columns if c.startswith("Q7_") and not c.endswith("_TEXT")]
if q7_cols:
    # converts values to numeric scores
    q7_num = df[q7_cols].apply(pd.to_numeric, errors='coerce')
    # renames labels with clearer names
    q7_num.columns = [f"Q7_{c.split('_',1)[1]}" for c in q7_num.columns]
    features = pd.concat([features, q7_num], axis=1)
    # adds mean
    features['Q7_mean'] = q7_num.mean(axis=1, skipna=True)

# Q10/Q11 will be used in gap analysis. Not included in features
# One-hot on Q13
if "Q13" in df.columns:
    # if Q13 is multichoice -> dummies, else get_dummies
    if df['Q13'].dropna().astype(str).str.contains(',').any():
        d13 = multiselect_dummies(df['Q13'], prefix='Q13')
    else:
        d13 = pd.get_dummies(df['Q13'].fillna('Missing'), prefix='Q13', dummy_na=False)
    features = pd.concat([features, d13], axis=1)

# Final control
print("Features shape:", features.shape)
print("Example columns:", list(features.columns)[:40])
# Saves a copy
features.to_csv(os.path.join(output, "features_Q5_Q13_raw.csv"), index=False)
features.head()

### 3.3 Preprocessing: imputation + scaling

In [ ]:
# If there aren't numeric features, skips
if features.shape[1] == 0:
    raise RuntimeError("No features generated: check that Q5–Q13 exist in dataframe.")

# imputes median per numeric
imp = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imp.fit_transform(features), columns=features.columns, index=features.index)

# scaling (need for KMeans)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X_imputed.columns, index=X_imputed.index)

# saves results
X_imputed.to_csv(os.path.join(output, "features_Q5_Q13_imputed.csv"), index=False)
X_scaled.to_csv(os.path.join(output, "features_Q5_Q13_scaled.csv"), index=False)
print("Imputation+scaling completed. Shape:", X_scaled.shape)

### 3.4 Cluster analysis (k-selection with silhouette + KMeans k* + profiles)

In [ ]:
# k preparation for best results
best_k = None
best_score = -1
sil_scores = {}
for k in range(2,7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    try:
        s = silhouette_score(X_scaled, labels)
    except Exception as e:
        s = np.nan
    sil_scores[k] = s
    if not np.isnan(s) and s > best_score:
        best_score = s; best_k = k

print("Silhouette scores for k:", sil_scores)
if best_k is None:
    best_k = 3  # fallback
    print("No valid silhouette: use k=3 (fallback).")
else:
    print(f"Selected k = {best_k} (silhouette = {best_score:.3f})")

# kmeans  fitting
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=30)
labels_final = km_final.fit_predict(X_scaled)
df['cluster'] = labels_final

# saves results
df[['cluster']].to_csv(os.path.join(output, "respondent_clusters.csv"), index=True)

# cluster profile: numeric features means + % per binary columns
profile_num = X_imputed.groupby(df['cluster']).mean().round(3)
profile_num.to_csv(os.path.join(output, "cluster_profile_numeric_means.csv"))

# only for 0/1 columns calculates % with value 1 per cluster
binary_cols = [c for c in X_imputed.columns if set(X_imputed[c].dropna().unique()).issubset({0,1})]
profile_bin = X_imputed[binary_cols].groupby(df['cluster']).mean().multiply(100).round(2)
profile_bin.to_csv(os.path.join(output, "cluster_profile_binary_pct.csv"))

print("Cluster dimension (counts):")
print(df['cluster'].value_counts())

# PCA 2D for visual
pca = PCA(n_components=2, random_state=42)
pc = pca.fit_transform(X_scaled)
plt.figure(figsize=(8,6))
for lab in np.unique(labels_final):
    sel = labels_final == lab
    plt.scatter(pc[sel,0], pc[sel,1], label=f"Cluster {lab}", alpha=0.7, s=40)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("2D PCA of respondent profiles (Q5–Q13 features)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output, "clusters_pca2d.png"), dpi=150)
plt.show()

### 3.5 Regression — (A) OLS on Q5_num, (B) Logistic (heavy user) 

In [ ]:
# linear regression with OLS

# takes y target
if 'Q5_num' not in features.columns:
    raise RuntimeError("Q5_num missing. Check that Q5 is present.")

# builds X_reg using same features
X_reg = X_imputed.copy()  # imputed but not scaled (OLS no scaling needed)
y_reg = features['Q5_num']

# aligns index and removes Nan rows in y
mask = y_reg.notna()
X_reg = X_reg.loc[mask]
y_reg = y_reg.loc[mask]

# add constant for intercept
X_reg_sm = sm.add_constant(X_reg)
model_ols = sm.OLS(y_reg.astype(float), X_reg_sm).fit(cov_type='HC3')
print(model_ols.summary())

# summary saving
with open(os.path.join(output, "regression_OLS_Q5_summary.txt"), "w") as f:
    f.write(model_ols.summary().as_text())

In [ ]:
# logistic regression (heavy user)

# binary target: heavy user = Q5_num (max score)
max_q5 = features['Q5_num'].max()
y_bin = (features['Q5_num'] == max_q5).astype(int)

X_log = X_imputed.loc[y_bin.index]

# delete elements with Var=0
keep = X_log.loc[:, X_log.nunique() > 1]

# logit with regularization L2
lr = LogisticRegression(max_iter=300, class_weight='balanced', solver='lbfgs')
lr.fit(keep, y_bin)

print("Accuracy:", lr.score(keep, y_bin))
print("Class report:\n", classification_report(y_bin, lr.predict(keep)))

# ordering coefficients by importance
coef_table = pd.DataFrame({
    "feature": keep.columns,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

# saves results
coef_table.to_csv(os.path.join(output, "logit_Q5_heavy_sklearn_coefs.csv"), index=False)
coef_table.head(20)


### 3.6 Gap analysis Q10 vs Q11 — Table & opportunity

In [ ]:
# empty dataset management
if "Q10" not in df.columns and "Q11" not in df.columns:
    print("Q10 and Q11 not present: no gap analysis.")
else:
    # new dummies
    df10 = multiselect_dummies(df["Q10"], prefix="Q10") if "Q10" in df.columns else pd.DataFrame(index=df.index)
    df11 = multiselect_dummies(df["Q11"], prefix="Q11") if "Q11" in df.columns else pd.DataFrame(index=df.index)

    if df10.shape[1] == 0 and df11.shape[1] == 0:
        print("No valid data in Q10 or Q11: no gap analysis.")
    else:
        # mapping
        cols10 = {c.replace("Q10_", ""): c for c in df10.columns}
        cols11 = {c.replace("Q11_", ""): c for c in df11.columns}

        # labels ordering
        all_labels = sorted(set(cols10.keys()).union(set(cols11.keys())))
        rows = []
        n_resp = df.shape[0]
        
        # gap analysis
        for lab in all_labels:
            col10 = cols10.get(lab)
            col11 = cols11.get(lab)
            cnt_not_used = int(df10[col10].sum()) if col10 else 0
            cnt_fund     = int(df11[col11].sum()) if col11 else 0
            pct_not_used = round(100 * cnt_not_used / n_resp, 2)
            pct_fund     = round(100 * cnt_fund / n_resp, 2)
            opp_score    = round((pct_not_used * pct_fund) / 100.0, 3)
            rows.append({
                "task": lab, "cnt_not_used": cnt_not_used, "pct_not_used": pct_not_used,
                "cnt_fund": cnt_fund, "pct_fund": pct_fund, "opportunity_score": opp_score
            })

        gap_df = pd.DataFrame(rows).sort_values("opportunity_score", ascending=False)
        gap_df.to_csv(os.path.join(output, "gap_Q10_vs_Q11_opportunity.csv"), index=False)

        display(gap_df.head(40))

        # scatter plot
        plt.figure(figsize=(8,6))
        plt.scatter(gap_df['pct_not_used'], gap_df['pct_fund'],
                    s=(gap_df['opportunity_score']*10 + 10), alpha=0.7)
        for _, r in gap_df.head(12).iterrows():
            plt.text(r['pct_not_used']+0.2, r['pct_fund']+0.2, r['task'], fontsize=8)
        plt.xlabel("% Not used (Q10)"); plt.ylabel("% Fundamental (Q11)")
        plt.title("Gap Analysis: Opportunity (Q10 vs Q11)")
        plt.grid(True); plt.tight_layout()
        plt.savefig(os.path.join(output, "gap_Q10_vs_Q11_scatter.png"), dpi=150)
        plt.show()